In [7]:
"""
STEP 1: PROBLEM STATEMENT SELECTION
=====================================
Project Title      : Yoga Pose Classification using MediaPipe Pose + ANN
Problem Statement  : Given an image of a person performing a yoga asana,
                      classify it into one of 6 pose categories.
Business Objective  : Enable automated, real-time posture recognition for
                      fitness/yoga apps to give users feedback on which
                      pose they are performing (foundation for future
                      form-correction features).
Expected Output     : Predicted pose label + confidence score, via a
                      Streamlit app supporting image upload and webcam.
Target Variable     : pose class -> {downdog, tree, warrior1,
                                      goddess, mountain, warrior2}
Input Features      : 33 MediaPipe Pose landmarks (x, y, z, visibility)
                      + 8 engineered joint angles (elbow, shoulder,
                      hip, knee - left & right)

STEP 2: DATASET COLLECTION
=====================================
Source   : Kaggle - "Yoga Pose Classification" by elysian01
           https://www.kaggle.com/datasets/elysian01/yoga-pose-classification
Type     : Existing dataset (image folders, pre-split train/test)
Classes  : downdog, tree, warrior1, goddess, mountain, warrior2
Notes    : Locally organized as yoga_set1 (downdog/tree/warrior1) and
           yoga_set2 (goddess/mountain/warrior2), each with train/test
           subfolders, then merged in Step 3 into one combined index.

Dataset requirements checklist:
  [x] Sufficient records   -> verified via class distribution in Step 3
  [x] Clean labels          -> folder-name-based labels, no free text
  [x] Well-defined target   -> single categorical column, 6 classes
"""

print(__doc__)


STEP 1: PROBLEM STATEMENT SELECTION
Project Title      : Yoga Pose Classification using MediaPipe Pose + ANN
Problem Statement  : Given an image of a person performing a yoga asana,
                      classify it into one of 6 pose categories.
Business Objective  : Enable automated, real-time posture recognition for
                      fitness/yoga apps to give users feedback on which
                      pose they are performing (foundation for future
                      form-correction features).
Expected Output     : Predicted pose label + confidence score, via a
                      Streamlit app supporting image upload and webcam.
Target Variable     : pose class -> {downdog, tree, warrior1,
                                      goddess, mountain, warrior2}
Input Features      : 33 MediaPipe Pose landmarks (x, y, z, visibility)
                      + 8 engineered joint angles (elbow, shoulder,
                      hip, knee - left & right)

STEP 2: DATASET COLLECTION
S

In [14]:
import os

BASE_DIR = r"C:\Users\Jagadeesh\Downloads\ANN_project\dataset\final_test"

print("Does BASE_DIR exist?", os.path.isdir(BASE_DIR))
print()

for set_name in ["yoga_set1", "yoga_set2"]:
    set_path = os.path.join(BASE_DIR, set_name)
    print(f"--- {set_name} ---")
    print("Exists:", os.path.isdir(set_path))
    if os.path.isdir(set_path):
        print("Contents:", os.listdir(set_path))
        for split in os.listdir(set_path):
            split_path = os.path.join(set_path, split)
            if os.path.isdir(split_path):
                print(f"  {split}/ ->", os.listdir(split_path))
    print()

Does BASE_DIR exist? True

--- yoga_set1 ---
Exists: True
Contents: ['1.jpg', '11.jpg', '3.jpg', '4.jpg', '5.jpg', '6.jpg', '7.jpg', '8.jpg', '9.jpg']

--- yoga_set2 ---
Exists: True
Contents: ['1.jpg', '10.jpg', '2.jpg', '3.jpg', '4.jpg', '5.jpg', '6.jpg', '7.jpg', '8.jpg', '9.jpg']



In [16]:
import sys
import pandas as pd

print(sys.executable)
print(pd.__version__)

c:\Users\Jagadeesh\AppData\Local\Programs\Python\Python310\python.exe
2.3.3


In [8]:
"""
STEP 3: DATASET IMPORT
Yoga Pose Classification using MediaPipe Pose + ANN

Folder structure expected (as per your VS Code tree):
final_test/
├── yoga_set1/
│   ├── train/{downdog, tree, warrior1}/*.jpg
│   └── test/{downdog, tree, warrior1}/*.jpg
└── yoga_set2/
    ├── train/{goddess, mountain, warrior2}/*.jpg
    └── test/{goddess, mountain, warrior2}/*.jpg
"""

import os
import pandas as pd
from PIL import Image

# ----------------------------------------------------------------
# Path to your dataset's "final_test" folder
# ----------------------------------------------------------------
BASE_DIR = r"C:\Users\Jagadeesh\Downloads\ANN_project\dataset"

SETS = {
    "yoga_set1": ["downdog", "tree", "warrior1"],
    "yoga_set2": ["goddess", "mountain", "warrior2"],
}
SPLITS = ["train", "test"]
VALID_EXT = (".jpg", ".jpeg", ".png")


def build_dataframe(base_dir: str) -> pd.DataFrame:
    """Walk both yoga_set folders and build one combined dataframe
    with columns: filepath, label, split, source_set."""
    records = []

    for set_name, classes in SETS.items():
        for split in SPLITS:
            for cls in classes:
                folder = os.path.join(base_dir, set_name, split, cls)
                if not os.path.isdir(folder):
                    print(f"[WARN] Missing folder: {folder}")
                    continue
                for fname in os.listdir(folder):
                    if fname.lower().endswith(VALID_EXT):
                        records.append({
                            "filepath": os.path.join(folder, fname),
                            "label": cls,
                            "split": split,
                            "source_set": set_name,
                        })

    df = pd.DataFrame(records)
    return df


def check_corrupt_images(df: pd.DataFrame):
    """Try opening every image; flag ones that fail to load."""
    bad_files = []
    for fp in df["filepath"]:
        try:
            with Image.open(fp) as img:
                img.verify()
        except Exception:
            bad_files.append(fp)
    return bad_files


df = build_dataframe(BASE_DIR)

print("=" * 60)
print("DATASET IMPORT SUMMARY")
print("=" * 60)
print(f"Total images found : {len(df)}")
print(f"Columns             : {list(df.columns)}")
print()

print("Class distribution (overall):")
print(df["label"].value_counts())
print()

print("Class distribution by split:")
print(df.groupby(["split", "label"]).size())
print()

print("Duplicate filepaths:", df["filepath"].duplicated().sum())
print()

print("Checking for corrupt / unreadable images (this may take a moment)...")
bad_files = check_corrupt_images(df)
print(f"Corrupt/unreadable images: {len(bad_files)}")
for f in bad_files[:20]:
    print("  -", f)

# Save the combined index so later steps don't need to re-scan folders
out_csv = os.path.join(os.getcwd(), "dataset_index.csv")
df.to_csv(out_csv, index=False)
print()
print(f"Saved combined dataset index -> {out_csv}")

DATASET IMPORT SUMMARY
Total images found : 2885
Columns             : ['filepath', 'label', 'split', 'source_set']

Class distribution (overall):
label
warrior2    564
goddess     484
mountain    483
downdog     477
tree        440
warrior1    437
Name: count, dtype: int64

Class distribution by split:
split  label   
test   downdog      96
       goddess      80
       mountain     30
       tree         69
       warrior1      5
       warrior2    109
train  downdog     381
       goddess     404
       mountain    453
       tree        371
       warrior1    432
       warrior2    455
dtype: int64

Duplicate filepaths: 0

Checking for corrupt / unreadable images (this may take a moment)...
Corrupt/unreadable images: 0

Saved combined dataset index -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\dataset_index.csv


In [9]:
"""
STEP 4: EXPLORATORY DATA ANALYSIS
Yoga Pose Classification using MediaPipe Pose + ANN

Reads dataset_index.csv (from Step 3) and produces:
  - Univariate: class distribution (count plot, pie chart)
  - Bivariate: class distribution by split (train vs test)
  - Multivariate: image dimension spread across classes
  - Sample image grid per class
  - Written observations
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

INDEX_CSV = os.path.join(os.getcwd(), "dataset_index.csv")
OUT_DIR = os.path.join(os.getcwd(), "eda_outputs")
os.makedirs(OUT_DIR, exist_ok=True)


def univariate_class_distribution(df):
    counts = df["label"].value_counts()

    fig, ax = plt.subplots(figsize=(8, 5))
    counts.plot(kind="bar", ax=ax, color="teal")
    ax.set_title("Univariate: Image Count per Pose Class")
    ax.set_xlabel("Pose")
    ax.set_ylabel("Number of Images")
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "class_distribution_bar.png"))
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 6))
    counts.plot(kind="pie", ax=ax, autopct="%1.1f%%")
    ax.set_ylabel("")
    ax.set_title("Class Share (Pie Chart)")
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "class_distribution_pie.png"))
    plt.close(fig)

    return counts


def bivariate_split_vs_class(df):
    ct = pd.crosstab(df["label"], df["split"])

    fig, ax = plt.subplots(figsize=(9, 5))
    ct.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title("Bivariate: Train vs Test Count per Class")
    ax.set_xlabel("Pose")
    ax.set_ylabel("Count")
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "split_vs_class.png"))
    plt.close(fig)

    return ct


def multivariate_image_dimensions(df, sample_per_class=30):
    """Sample a few images per class and record width/height to spot
    resolution inconsistencies (important since two sets were merged)."""
    records = []
    for cls, group in df.groupby("label"):
        sample = group.sample(min(sample_per_class, len(group)), random_state=42)
        for fp in sample["filepath"]:
            try:
                with Image.open(fp) as img:
                    w, h = img.size
                records.append({"label": cls, "width": w, "height": h})
            except Exception:
                continue

    dim_df = pd.DataFrame(records)

    fig, ax = plt.subplots(figsize=(8, 5))
    for cls in dim_df["label"].unique():
        subset = dim_df[dim_df["label"] == cls]
        ax.scatter(subset["width"], subset["height"], label=cls, alpha=0.6)
    ax.set_title("Multivariate: Image Width vs Height by Class")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "image_dimensions_scatter.png"))
    plt.close(fig)

    return dim_df


def sample_image_grid(df, n_per_class=3):
    classes = sorted(df["label"].unique())
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(n_per_class * 2.5, len(classes) * 2.5))

    for row, cls in enumerate(classes):
        sample = df[df["label"] == cls].sample(min(n_per_class, len(df[df["label"] == cls])), random_state=1)
        for col in range(n_per_class):
            ax = axes[row, col] if len(classes) > 1 else axes[col]
            ax.axis("off")
            if col < len(sample):
                fp = sample.iloc[col]["filepath"]
                try:
                    img = Image.open(fp)
                    ax.imshow(img)
                except Exception:
                    pass
            if col == 0:
                ax.set_ylabel(cls, fontsize=9)

    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "sample_image_grid.png"))
    plt.close(fig)


df = pd.read_csv(INDEX_CSV)

print("=" * 60)
print("STEP 4: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

counts = univariate_class_distribution(df)
print("\nClass counts:\n", counts)

ct = bivariate_split_vs_class(df)
print("\nTrain/Test split per class:\n", ct)

dim_df = multivariate_image_dimensions(df)
print("\nImage dimension summary:\n", dim_df.groupby("label")[["width", "height"]].describe())

sample_image_grid(df)

print(f"\nAll plots saved to: {OUT_DIR}")

# ---- Written observations (fill in after reviewing the plots) ----
print("""
OBSERVATIONS (edit based on actual plots):
- Note whether classes are balanced or if any pose has noticeably
  fewer images (common after merging two Kaggle sets).
- Note whether yoga_set1 and yoga_set2 images differ in resolution
  or aspect ratio - this affects the resize target in Step 5.
- Note whether any pose images are cropped/occluded in a way that
  might cause MediaPipe to fail landmark detection in Step 6.

INSIGHTS / RECOMMENDATIONS:
- If classes are imbalanced, consider class_weight in the ANN (Step 9)
  or oversampling the minority class.
- Standardize all images to one resize target before feature extraction.
""")

STEP 4: EXPLORATORY DATA ANALYSIS

Class counts:
 label
warrior2    564
goddess     484
mountain    483
downdog     477
tree        440
warrior1    437
Name: count, dtype: int64

Train/Test split per class:
 split     test  train
label                
downdog     96    381
goddess     80    404
mountain    30    453
tree        69    371
warrior1     5    432
warrior2   109    455

Image dimension summary:
          width                                                           \
         count         mean         std    min     25%     50%      75%   
label                                                                     
downdog   30.0  1114.300000  683.201952  205.0  751.25  1080.0  1080.00   
goddess   30.0   592.333333  307.046367  183.0  305.00   522.5   755.25   
mountain  30.0   813.500000  693.092309  222.0  400.00   720.0  1080.00   
tree      30.0   873.033333  565.875061  183.0  387.25   980.5  1080.00   
warrior1  30.0  1127.466667  707.450485  275.0  852.25  1080.0  

In [10]:
"""
STEP 5: DATA PREPROCESSING (Image Project)
Yoga Pose Classification using MediaPipe Pose + ANN

Since Step 6 extracts MediaPipe landmarks (not raw pixels) as the
model input, heavy pixel-level preprocessing isn't required for the
ANN itself. This step still standardizes images before landmark
extraction so MediaPipe gets consistent input, and offers an
augmentation utility if you want to expand a class with too few images.
"""

import os
import cv2
import pandas as pd

INDEX_CSV = os.path.join(os.getcwd(), "dataset_index.csv")
RESIZE_DIM = (640, 640)  # MediaPipe works fine on this; adjust if needed


def resize_and_check(filepath, target_dim=RESIZE_DIM):
    """Load an image, resize it, return the resized array (or None
    if the file can't be read). This is the same resize you should
    apply consistently before Step 6 if you pre-resize on disk."""
    img = cv2.imread(filepath)
    if img is None:
        return None
    resized = cv2.resize(img, target_dim, interpolation=cv2.INTER_AREA)
    return resized


def normalize(img):
    """Scale pixel values to [0, 1]. Only needed if you later train a
    CNN directly on pixels instead of / in addition to MediaPipe
    landmarks."""
    return img.astype("float32") / 255.0


def augment_image(img):
    """Basic augmentation: horizontal flip + slight brightness jitter.
    Use only for classes that came up short in Step 4's EDA - don't
    augment MediaPipe landmark features directly, augment the source
    image and re-run Step 6 on the augmented copy."""
    flipped = cv2.flip(img, 1)
    bright = cv2.convertScaleAbs(img, alpha=1.0, beta=15)
    return [flipped, bright]


def audit_preprocessing(df: pd.DataFrame, sample_size=50):
    """Quick pass over a sample to confirm resize works cleanly and
    report how many files fail to load."""
    sample = df.sample(min(sample_size, len(df)), random_state=42)
    failures = []
    for fp in sample["filepath"]:
        if resize_and_check(fp) is None:
            failures.append(fp)

    print(f"Checked {len(sample)} images, {len(failures)} failed to load/resize.")
    for f in failures[:10]:
        print("  -", f)


df = pd.read_csv(INDEX_CSV)
print("=" * 60)
print("STEP 5: DATA PREPROCESSING AUDIT")
print("=" * 60)
audit_preprocessing(df)
print(f"\nTarget resize dimension for all images: {RESIZE_DIM}")
print("Note: Step 6 (MediaPipe) reads images directly and works on")
print("original resolution internally - resizing here is mainly to")
print("confirm no files are corrupt and to standardize before any")
print("optional CNN-based feature path.")

STEP 5: DATA PREPROCESSING AUDIT
Checked 50 images, 0 failed to load/resize.

Target resize dimension for all images: (640, 640)
Note: Step 6 (MediaPipe) reads images directly and works on
original resolution internally - resizing here is mainly to
confirm no files are corrupt and to standardize before any
optional CNN-based feature path.


In [2]:
"""
STEP 6: FEATURE EXTRACTION (MediaPipe Landmarks)
Yoga Pose Classification using MediaPipe Pose + ANN

Reads dataset_index.csv (produced by Step 3), runs MediaPipe Pose on
every image, extracts the 33 landmarks, engineers joint-angle
features, and saves a clean feature table ready for Step 7.
"""

import os
import math
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

mp_pose = mp.solutions.pose

INDEX_CSV = os.path.join(os.getcwd(), "dataset_index.csv")
OUT_CSV = os.path.join(os.getcwd(), "pose_features.csv")
MIN_DETECTION_CONFIDENCE = 0.5


def calculate_angle(a, b, c):
    """Angle at point b, given three (x, y) points a-b-c, in degrees."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = math.atan2(c[1] - b[1], c[0] - b[0]) - math.atan2(a[1] - b[1], a[0] - b[0])
    angle = abs(radians * 180.0 / math.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle


def extract_landmarks(image_path, pose):
    """Run MediaPipe Pose on one image, return dict of landmark
    coords + engineered joint angles, or None if no pose detected."""
    image = cv2.imread(image_path)
    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if not results.pose_landmarks:
        return None

    lm = results.pose_landmarks.landmark
    coords = {}
    for i, point in enumerate(lm):
        coords[f"x{i}"] = point.x
        coords[f"y{i}"] = point.y
        coords[f"z{i}"] = point.z
        coords[f"v{i}"] = point.visibility

    def pt(idx):
        return (lm[idx].x, lm[idx].y)

    L_SHOULDER, R_SHOULDER = 11, 12
    L_ELBOW, R_ELBOW = 13, 14
    L_WRIST, R_WRIST = 15, 16
    L_HIP, R_HIP = 23, 24
    L_KNEE, R_KNEE = 25, 26
    L_ANKLE, R_ANKLE = 27, 28

    angles = {
        "left_elbow_angle": calculate_angle(pt(L_SHOULDER), pt(L_ELBOW), pt(L_WRIST)),
        "right_elbow_angle": calculate_angle(pt(R_SHOULDER), pt(R_ELBOW), pt(R_WRIST)),
        "left_shoulder_angle": calculate_angle(pt(L_ELBOW), pt(L_SHOULDER), pt(L_HIP)),
        "right_shoulder_angle": calculate_angle(pt(R_ELBOW), pt(R_SHOULDER), pt(R_HIP)),
        "left_hip_angle": calculate_angle(pt(L_SHOULDER), pt(L_HIP), pt(L_KNEE)),
        "right_hip_angle": calculate_angle(pt(R_SHOULDER), pt(R_HIP), pt(R_KNEE)),
        "left_knee_angle": calculate_angle(pt(L_HIP), pt(L_KNEE), pt(L_ANKLE)),
        "right_knee_angle": calculate_angle(pt(R_HIP), pt(R_KNEE), pt(R_ANKLE)),
    }

    features = {**coords, **angles}
    return features


df_index = pd.read_csv(INDEX_CSV)
print(f"Loaded {len(df_index)} image entries from {INDEX_CSV}")

rows = []
failed = []

with mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
) as pose:
    for i, row in df_index.iterrows():
        feats = extract_landmarks(row["filepath"], pose)
        if feats is None:
            failed.append(row["filepath"])
            continue

        feats["label"] = row["label"]
        feats["split"] = row["split"]
        feats["source_set"] = row["source_set"]
        feats["filepath"] = row["filepath"]
        rows.append(feats)

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(df_index)}...")

df_features = pd.DataFrame(rows)
df_features.to_csv(OUT_CSV, index=False)

print("=" * 60)
print("FEATURE EXTRACTION SUMMARY")
print("=" * 60)
print(f"Successfully processed : {len(df_features)}")
print(f"Failed (no pose found) : {len(failed)}")
if failed:
    print("Sample failed files:")
    for f in failed[:10]:
        print("  -", f)
print(f"Saved feature table -> {OUT_CSV}")


Loaded 2885 image entries from c:\Users\Jagadeesh\Downloads\ANN_project\dataset\dataset_index.csv


c:\Users\Jagadeesh\Downloads\ANN_project\yoga_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processed 100/2885...
Processed 200/2885...
Processed 300/2885...
Processed 400/2885...
Processed 500/2885...
Processed 600/2885...
Processed 700/2885...
Processed 800/2885...
Processed 900/2885...
Processed 1000/2885...
Processed 1100/2885...
Processed 1200/2885...
Processed 1300/2885...
Processed 1400/2885...
Processed 1500/2885...
Processed 1600/2885...
Processed 1700/2885...
Processed 1800/2885...
Processed 1900/2885...
Processed 2000/2885...
Processed 2100/2885...
Processed 2200/2885...
Processed 2300/2885...
Processed 2400/2885...
Processed 2500/2885...
Processed 2600/2885...
Processed 2700/2885...
Processed 2800/2885...
FEATURE EXTRACTION SUMMARY
Successfully processed : 2794
Failed (no pose found) : 91
Sample failed files:
  - C:\Users\Jagadeesh\Downloads\ANN_project\dataset\yoga_set1\train\downdog\116.jpg
  - C:\Users\Jagadeesh\Downloads\ANN_project\dataset\yoga_set1\train\downdog\133.jpg
  - C:\Users\Jagadeesh\Downloads\ANN_project\dataset\yoga_set1\train\downdog\136.jpg
  - 

In [3]:
"""
STEP 7: INPUT / OUTPUT SEPARATION
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder

FEATURES_CSV = os.path.join(os.getcwd(), "pose_features.csv")
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

NON_FEATURE_COLS = ["label", "split", "source_set", "filepath"]


def separate_X_y(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].copy()
    y_raw = df["label"].copy()
    return X, y_raw, feature_cols


def encode_labels(y_raw: pd.Series):
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y_raw)
    return y_encoded, encoder


df = pd.read_csv(FEATURES_CSV)

print("=" * 60)
print("STEP 7: INPUT / OUTPUT SEPARATION")
print("=" * 60)

X, y_raw, feature_cols = separate_X_y(df)
y_encoded, encoder = encode_labels(y_raw)

print(f"X shape: {X.shape}")
print(f"Number of feature columns: {len(feature_cols)}")
print(f"Classes found: {list(encoder.classes_)}")
print(f"Encoded label mapping: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

df_out = X.copy()
df_out["label_encoded"] = y_encoded
df_out["label"] = y_raw.values
df_out["split"] = df["split"].values

out_csv = os.path.join(os.getcwd(), "features_encoded.csv")
df_out.to_csv(out_csv, index=False)
joblib.dump(encoder, os.path.join(ARTIFACT_DIR, "label_encoder.pkl"))

print(f"\nSaved encoded feature table -> {out_csv}")
print(f"Saved label encoder -> {os.path.join(ARTIFACT_DIR, 'label_encoder.pkl')}")

STEP 7: INPUT / OUTPUT SEPARATION
X shape: (2794, 140)
Number of feature columns: 140
Classes found: ['downdog', 'goddess', 'mountain', 'tree', 'warrior1', 'warrior2']
Encoded label mapping: {'downdog': 0, 'goddess': 1, 'mountain': 2, 'tree': 3, 'warrior1': 4, 'warrior2': 5}

Saved encoded feature table -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\features_encoded.csv
Saved label encoder -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\artifacts\label_encoder.pkl


In [4]:
"""
STEP 8: TRAIN / TEST SPLIT
STEP 8a: FEATURE SCALING
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import numpy as np
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

FEATURES_CSV = os.path.join(os.getcwd(), "features_encoded.csv")
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

NON_FEATURE_COLS = ["label", "label_encoded", "split"]


def split_by_existing_folder(df: pd.DataFrame):
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)

    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

    X_train = train_df[feature_cols]
    y_train = train_df["label_encoded"]
    X_test = test_df[feature_cols]
    y_test = test_df["label_encoded"]

    return X_train, X_test, y_train, y_test, feature_cols


def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


df = pd.read_csv(FEATURES_CSV)

print("=" * 60)
print("STEP 8: TRAIN / TEST SPLIT (existing folder-based split)")
print("=" * 60)

X_train, X_test, y_train, y_test, feature_cols = split_by_existing_folder(df)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance:\n{y_train.value_counts()}")
print(f"Test class balance:\n{y_test.value_counts()}")

print("\nSTEP 8a: FEATURE SCALING")
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
print(f"Scaled train shape: {X_train_scaled.shape}")

joblib.dump(scaler, os.path.join(ARTIFACT_DIR, "scaler.pkl"))
joblib.dump(feature_cols, os.path.join(ARTIFACT_DIR, "feature_cols.pkl"))

np.save(os.path.join(ARTIFACT_DIR, "X_train.npy"), X_train_scaled)
np.save(os.path.join(ARTIFACT_DIR, "X_test.npy"), X_test_scaled)
np.save(os.path.join(ARTIFACT_DIR, "y_train.npy"), y_train.values)
np.save(os.path.join(ARTIFACT_DIR, "y_test.npy"), y_test.values)

print(f"\nSaved scaler + train/test arrays to {ARTIFACT_DIR}")

STEP 8: TRAIN / TEST SPLIT (existing folder-based split)
Train shape: (2410, 140), Test shape: (384, 140)
Train class balance:
label_encoded
5    451
2    436
4    419
1    391
0    357
3    356
Name: count, dtype: int64
Test class balance:
label_encoded
5    107
0     94
1     80
3     69
2     30
4      4
Name: count, dtype: int64

STEP 8a: FEATURE SCALING
Scaled train shape: (2410, 140)

Saved scaler + train/test arrays to c:\Users\Jagadeesh\Downloads\ANN_project\dataset\artifacts


In [5]:
"""
STEP 9: MODEL BUILDING (ANN Baseline)
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import numpy as np
import joblib
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")


def load_split():
    X_train = np.load(os.path.join(ARTIFACT_DIR, "X_train.npy"))
    X_test = np.load(os.path.join(ARTIFACT_DIR, "X_test.npy"))
    y_train = np.load(os.path.join(ARTIFACT_DIR, "y_train.npy"))
    y_test = np.load(os.path.join(ARTIFACT_DIR, "y_test.npy"))
    return X_train, X_test, y_train, y_test


def build_ann(input_dim, num_classes, units=(128, 64), dropout=0.3, lr=1e-3):
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for u in units:
        model.add(layers.Dense(u, activation="relu"))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


X_train, X_test, y_train, y_test = load_split()

encoder = joblib.load(os.path.join(ARTIFACT_DIR, "label_encoder.pkl"))
num_classes = len(encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("=" * 60)
print("STEP 9: ANN BASELINE MODEL")
print("=" * 60)
print(f"Input dim: {X_train.shape[1]}, Classes: {num_classes}")

model = build_ann(input_dim=X_train.shape[1], num_classes=num_classes)
model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

val_loss, val_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nBaseline ANN - Test Accuracy: {val_acc:.4f}, Test Loss: {val_loss:.4f}")

model.save(os.path.join(ARTIFACT_DIR, "ann_baseline.keras"))
print(f"Saved baseline model -> {os.path.join(ARTIFACT_DIR, 'ann_baseline.keras')}")

STEP 9: ANN BASELINE MODEL
Input dim: 140, Classes: 6

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               18048     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 6)                 390       
                                                                 
Total params: 26694 (104.27 KB)
Trainable params: 26694 (104.27 KB)
Non-trainable params: 0 (0.00 Byte)
_____________________________

In [6]:
"""
STEP 10: HYPERPARAMETER TUNING (Optuna)
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import json
import numpy as np
import joblib
import optuna
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
N_TRIALS = 30


def load_split():
    X_train = np.load(os.path.join(ARTIFACT_DIR, "X_train.npy"))
    X_test = np.load(os.path.join(ARTIFACT_DIR, "X_test.npy"))
    y_train = np.load(os.path.join(ARTIFACT_DIR, "y_train.npy"))
    y_test = np.load(os.path.join(ARTIFACT_DIR, "y_test.npy"))
    return X_train, X_test, y_train, y_test


def build_model(trial, input_dim, num_classes):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for i in range(n_layers):
        units = trial.suggest_int(f"units_l{i}", 32, 256, step=32)
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def objective(trial, X_train, y_train_cat, X_test, y_test_cat, input_dim, num_classes):
    model = build_model(trial, input_dim, num_classes)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train_cat,
        validation_data=(X_test, y_test_cat),
        epochs=60,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0,
    )

    val_acc = max(history.history["val_accuracy"])
    return val_acc


X_train, X_test, y_train, y_test = load_split()
encoder = joblib.load(os.path.join(ARTIFACT_DIR, "label_encoder.pkl"))
num_classes = len(encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("=" * 60)
print("STEP 10: OPTUNA HYPERPARAMETER TUNING")
print("=" * 60)

study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: objective(
        trial, X_train, y_train_cat, X_test, y_test_cat,
        input_dim=X_train.shape[1], num_classes=num_classes
    ),
    n_trials=N_TRIALS,
)

print(f"\nBest trial value (val_accuracy): {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

with open(os.path.join(ARTIFACT_DIR, "best_params.json"), "w") as f:
    json.dump(study.best_params, f, indent=2)

print(f"Saved best params -> {os.path.join(ARTIFACT_DIR, 'best_params.json')}")

best_model = build_model(
    optuna.trial.FixedTrial(study.best_params),
    input_dim=X_train.shape[1],
    num_classes=num_classes,
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
best_model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)
best_model.save(os.path.join(ARTIFACT_DIR, "ann_tuned.keras"))
print(f"Saved tuned model -> {os.path.join(ARTIFACT_DIR, 'ann_tuned.keras')}")

c:\Users\Jagadeesh\Downloads\ANN_project\yoga_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-13 19:10:45,360] A new study created in memory with name: no-name-cd1b5fd8-b811-47d3-90bb-ba92eac42843


STEP 10: OPTUNA HYPERPARAMETER TUNING


[I 2026-08-13 19:10:55,475] Trial 0 finished with value: 0.96875 and parameters: {'n_layers': 1, 'dropout': 0.2762089618090048, 'lr': 0.0003699943763674047, 'units_l0': 256}. Best is trial 0 with value: 0.96875.
[I 2026-08-13 19:10:59,963] Trial 1 finished with value: 0.9661458134651184 and parameters: {'n_layers': 3, 'dropout': 0.29826924829916157, 'lr': 0.0049894927470826795, 'units_l0': 192, 'units_l1': 160, 'units_l2': 32}. Best is trial 0 with value: 0.96875.
[I 2026-08-13 19:11:10,322] Trial 2 finished with value: 0.96875 and parameters: {'n_layers': 2, 'dropout': 0.37826944327407186, 'lr': 0.0004302013148217497, 'units_l0': 64, 'units_l1': 64}. Best is trial 0 with value: 0.96875.
[I 2026-08-13 19:11:20,309] Trial 3 finished with value: 0.9713541865348816 and parameters: {'n_layers': 1, 'dropout': 0.1855674898409103, 'lr': 0.00047724432846087264, 'units_l0': 64}. Best is trial 3 with value: 0.9713541865348816.
[I 2026-08-13 19:11:29,584] Trial 4 finished with value: 0.9713541865


Best trial value (val_accuracy): 0.9766
Best params: {'n_layers': 1, 'dropout': 0.10784586501751381, 'lr': 0.0036850022427605818, 'units_l0': 224}
Saved best params -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\artifacts\best_params.json
Epoch 1/100
76/76 [==============================] - 1s 5ms/step - loss: 0.4650 - accuracy: 0.8672 - val_loss: 0.2293 - val_accuracy: 0.9323
Epoch 2/100
76/76 [==============================] - 0s 3ms/step - loss: 0.2125 - accuracy: 0.9423 - val_loss: 0.2521 - val_accuracy: 0.9557
Epoch 3/100
76/76 [==============================] - 0s 3ms/step - loss: 0.1757 - accuracy: 0.9535 - val_loss: 0.2079 - val_accuracy: 0.9479
Epoch 4/100
76/76 [==============================] - 0s 3ms/step - loss: 0.1699 - accuracy: 0.9560 - val_loss: 0.2196 - val_accuracy: 0.9583
Epoch 5/100
76/76 [==============================] - 0s 3ms/step - loss: 0.1387 - accuracy: 0.9627 - val_loss: 0.2037 - val_accuracy: 0.9688
Epoch 6/100
76/76 [==============================]

In [7]:
"""
STEP 11: MODEL EVALUATION
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from tensorflow.keras.utils import to_categorical

ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
OUT_DIR = os.path.join(os.getcwd(), "eval_outputs")
os.makedirs(OUT_DIR, exist_ok=True)


def load_split():
    X_test = np.load(os.path.join(ARTIFACT_DIR, "X_test.npy"))
    y_test = np.load(os.path.join(ARTIFACT_DIR, "y_test.npy"))
    return X_test, y_test


X_test, y_test = load_split()
encoder = joblib.load(os.path.join(ARTIFACT_DIR, "label_encoder.pkl"))
class_names = list(encoder.classes_)
num_classes = len(class_names)

model_path = os.path.join(ARTIFACT_DIR, "ann_tuned.keras")
if not os.path.exists(model_path):
    model_path = os.path.join(ARTIFACT_DIR, "ann_baseline.keras")
model = tf.keras.models.load_model(model_path)
print(f"Evaluating model: {model_path}")

y_test_cat = to_categorical(y_test, num_classes=num_classes)
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("=" * 60)
print("STEP 11: MODEL EVALUATION")
print("=" * 60)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")
try:
    roc_auc = roc_auc_score(y_test_cat, y_pred_probs, multi_class="ovr")
except ValueError:
    roc_auc = float("nan")

print(f"Accuracy       : {acc:.4f}")
print(f"Precision(macro): {precision:.4f}")
print(f"Recall(macro)   : {recall:.4f}")
print(f"F1 Score(macro) : {f1:.4f}")
print(f"ROC-AUC(ovr)    : {roc_auc:.4f}")

print("\nClassification Report:")
report = classification_report(y_test, y_pred, target_names=class_names)
print(report)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"))
plt.close(fig)

print(f"\nSaved classification report + confusion matrix -> {OUT_DIR}")

Evaluating model: c:\Users\Jagadeesh\Downloads\ANN_project\dataset\artifacts\ann_tuned.keras
12/12 [==============================] - 0s 2ms/step
STEP 11: MODEL EVALUATION
Accuracy       : 0.9635
Precision(macro): 0.9133
Recall(macro)   : 0.8881
F1 Score(macro) : 0.8985
ROC-AUC(ovr)    : 0.9943

Classification Report:
              precision    recall  f1-score   support

     downdog       1.00      1.00      1.00        94
     goddess       0.94      0.96      0.95        80
    mountain       0.94      0.97      0.95        30
        tree       0.98      0.93      0.96        69
    warrior1       0.67      0.50      0.57         4
    warrior2       0.95      0.97      0.96       107

    accuracy                           0.96       384
   macro avg       0.91      0.89      0.90       384
weighted avg       0.96      0.96      0.96       384


Saved classification report + confusion matrix -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\eval_outputs


In [8]:
"""
STEP 12: MODEL SAVING
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import shutil
import joblib
import tensorflow as tf

ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
FINAL_DIR = os.path.join(os.getcwd(), "final_model")
os.makedirs(FINAL_DIR, exist_ok=True)

print("=" * 60)
print("STEP 12: MODEL SAVING")
print("=" * 60)

tuned_path = os.path.join(ARTIFACT_DIR, "ann_tuned.keras")
baseline_path = os.path.join(ARTIFACT_DIR, "ann_baseline.keras")
source_model = tuned_path if os.path.exists(tuned_path) else baseline_path

final_model_path = os.path.join(FINAL_DIR, "yoga_pose_model.keras")
shutil.copy(source_model, final_model_path)
print(f"Copied model: {source_model} -> {final_model_path}")

for fname in ["scaler.pkl", "label_encoder.pkl", "feature_cols.pkl"]:
    src = os.path.join(ARTIFACT_DIR, fname)
    dst = os.path.join(FINAL_DIR, fname)
    shutil.copy(src, dst)
    print(f"Copied: {fname}")

print(f"\nAll final artifacts saved in: {FINAL_DIR}")
print("Contents:")
for f in os.listdir(FINAL_DIR):
    print("  -", f)

STEP 12: MODEL SAVING
Copied model: c:\Users\Jagadeesh\Downloads\ANN_project\dataset\artifacts\ann_tuned.keras -> c:\Users\Jagadeesh\Downloads\ANN_project\dataset\final_model\yoga_pose_model.keras
Copied: scaler.pkl
Copied: label_encoder.pkl
Copied: feature_cols.pkl

All final artifacts saved in: c:\Users\Jagadeesh\Downloads\ANN_project\dataset\final_model
Contents:
  - feature_cols.pkl
  - label_encoder.pkl
  - scaler.pkl
  - yoga_pose_model.keras


In [11]:
"""
STEP 12a: MEDIAPIPE INTEGRATION
Yoga Pose Classification using MediaPipe Pose + ANN
"""

import os
import math
import cv2
import numpy as np
import joblib
import mediapipe as mp
import tensorflow as tf

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

FINAL_DIR = os.path.join(os.getcwd(), "final_model")


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = math.atan2(c[1] - b[1], c[0] - b[0]) - math.atan2(a[1] - b[1], a[0] - b[0])
    angle = abs(radians * 180.0 / math.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle


class YogaPosePredictor:
    def __init__(self, model_dir=FINAL_DIR):
        self.model = tf.keras.models.load_model(os.path.join(model_dir, "yoga_pose_model.keras"))
        self.scaler = joblib.load(os.path.join(model_dir, "scaler.pkl"))
        self.encoder = joblib.load(os.path.join(model_dir, "label_encoder.pkl"))
        self.feature_cols = joblib.load(os.path.join(model_dir, "feature_cols.pkl"))
        self.pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

    def extract_features(self, image_bgr):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        results = self.pose.process(image_rgb)

        annotated = image_bgr.copy()

        if not results.pose_landmarks:
            return None, annotated, False

        mp_drawing.draw_landmarks(
            annotated, results.pose_landmarks, mp_pose.POSE_CONNECTIONS
        )

        lm = results.pose_landmarks.landmark
        coords = {}
        for i, point in enumerate(lm):
            coords[f"x{i}"] = point.x
            coords[f"y{i}"] = point.y
            coords[f"z{i}"] = point.z
            coords[f"v{i}"] = point.visibility

        def pt(idx):
            return (lm[idx].x, lm[idx].y)

        L_SHOULDER, R_SHOULDER = 11, 12
        L_ELBOW, R_ELBOW = 13, 14
        L_WRIST, R_WRIST = 15, 16
        L_HIP, R_HIP = 23, 24
        L_KNEE, R_KNEE = 25, 26
        L_ANKLE, R_ANKLE = 27, 28

        angles = {
            "left_elbow_angle": calculate_angle(pt(L_SHOULDER), pt(L_ELBOW), pt(L_WRIST)),
            "right_elbow_angle": calculate_angle(pt(R_SHOULDER), pt(R_ELBOW), pt(R_WRIST)),
            "left_shoulder_angle": calculate_angle(pt(L_ELBOW), pt(L_SHOULDER), pt(L_HIP)),
            "right_shoulder_angle": calculate_angle(pt(R_ELBOW), pt(R_SHOULDER), pt(R_HIP)),
            "left_hip_angle": calculate_angle(pt(L_SHOULDER), pt(L_HIP), pt(L_KNEE)),
            "right_hip_angle": calculate_angle(pt(R_SHOULDER), pt(R_HIP), pt(R_KNEE)),
            "left_knee_angle": calculate_angle(pt(L_HIP), pt(L_KNEE), pt(L_ANKLE)),
            "right_knee_angle": calculate_angle(pt(R_HIP), pt(R_KNEE), pt(R_ANKLE)),
        }

        features = {**coords, **angles}
        return features, annotated, True

    def predict(self, image_bgr):
        features, annotated, found = self.extract_features(image_bgr)
        if not found:
            return None, None, annotated

        X = np.array([[features[c] for c in self.feature_cols]])
        X_scaled = self.scaler.transform(X)

        probs = self.model.predict(X_scaled, verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        label = self.encoder.inverse_transform([pred_idx])[0]
        confidence = float(probs[pred_idx])

        return label, confidence, annotated